# EDA/SDA 1: Field Comparison and Scope

PredictLeads × TripleTen externship · Jacob Colombo

## Purpose

Clean PredictLeads' full job-openings file (every listing first seen March 30 – September 30, 2026), compare fields, cut to the scope (**the roles TripleTen trains for, plus adjacent tech roles**) and explore it.

- **Q1 (section 3):** how do computer and mathematical roles compare with every other field?
- **Q2 (section 5):** which factors should drive prioritization for a career changer?
- **Q3 (section 6):** what time information can prioritization rely on?

Fix or flag, never drop, except data errors; aggregates only, never rows. **"Still open"** means last seen within 2 days of September 30.

**Hands off (section 7):** the scope table (every field) and its descriptions, the occupation summary, and company activity.

| Notebook | Question | Product |
|:--|:--|:--|
| **EDA_SDA_1** | **What's in the data, and what's the scope?** | **The scope table** |
| EDA_SDA_2 | Which scope listings do we keep, and how do they behave? | Cleaned and flagged scope table (ghosts, hiding rules) |
| EDA_SDA_3 | What do the descriptions say? | Text features and O*NET skills |
| EDA_SDA_4 | Which features hold up? | `modeling_ready.parquet` (the features that survived) |
| TimeSeries | When do listings appear and close? | Timing patterns by month |
| Profiles | Who really changed into or within tech or math, and how? (frozen) | The 2,813 career changers |
| ML_Fit | Which roles does a career changer fit? | The trained layer (role fit), saved for the app |
| ML_Rank | Which listings should they apply to first? | Listing scores and the queue's weights, saved for the app |
| Evaluate | How well do the app's rankers do on real career moves? | The comparison: `rrf`, `mlrank`, combined |

Same table as [`../PROJECT.md`](../PROJECT.md) ("The notebooks"), kept in step with it.

## 1. Load and build

1b–1d show the raw structure on a 200,000-document sample, then flatten it (`data/flat/`). 1e builds the full file (13.4M listings) from `data/raw/` (`python -m fetch.get_predictleads`) or S3 (`src/stream_full.py`). `BUILD_MODE`: **bulk** (about 35 minutes, 16 GB of RAM), **stream** (a few hours, for an 8 GB machine) or **auto**; both build the same tables, and finished steps are skipped.

In [ ]:
# 1a Libraries, the data loader (src/pl_data.py), the full-file rows (src/rows_table.py) and the shared helpers
import gc
import json
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

PROJECT_ROOT = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "src" / "eda_helpers.py").exists())
sys.path.insert(0, str(PROJECT_ROOT))  # so `src` can be imported from notebooks/
from src.pl_data import DATA_DIR, FLAT_DIR, describe_structure, flatten_all, load_flat, load_raw
pd.set_option("display.max_rows", 200)  # long summary tables (the feature dictionary) show in full
pd.set_option("display.max_colwidth", 140)
_marks = [time.time()]
def section_done(label):
    """Print how long a section took and the total so far, so a long run shows its progress."""
    now = time.time()
    print(f"[{label} done in {(now - _marks[-1]) / 60:.1f} min; {(now - _marks[0]) / 60:.1f} min in total]")
    _marks.append(now)

# Summary tables (show, show_shaded), compact counts, charts in one style (fixed color order: blue, orange, aqua, yellow)
# and Kaplan–Meier survival, shared with the other EDA notebooks
from src.eda_helpers import (AQUA, BLUE, INK, INK_2, MUTED, ORANGE, SURFACE, bar_h, compact, finish, km_curve,
                             km_median, km_survival, pct, show, show_shaded, surv_at, use_chart_style)
from src.rows_table import (bit_counts, cat_contains, cat_map, load_rows, relabel, row_extras, save_final,
                           with_missing)
use_chart_style()

In [ ]:
# 1b Load the raw documents ({} when every table is already flattened)
raw = load_raw()

In [ ]:
# 1c Column names and the Python type of each value in the first row
describe_structure(raw)

In [ ]:
# 1d Flatten all four tables (skipped when already done)
flatten_all(raw)

In [ ]:
# 1e Build the full-file tables (skipped when done)
BUILD_MODE = "auto"  # "bulk": one parallel pass | "stream": three memory-light passes | "auto": bulk when the machine can take it
from src.stream_full import build
build(BUILD_MODE)

## 2. Data cleaning

Full-file fixes (13.4M listings), data quality only: nothing is removed but data errors. The cleaning report (2f) lists every action.

- **Text** becomes counts and flags in the stream (`src/listing_columns.py`). Without a description a text flag is unknown, not False.
- **Dates:** impossible timelines are removed; implausible `posted` dates are blanked, and the listing stays.
- **Reposts** are kept and flagged, with features built from earlier copies only. Duplicate ids are dropped.
- **Pay** becomes one hourly rate; reversed ranges are swapped, wrong units re-read, and values outside $10k–$1M a year blanked.
- **Derived columns:** `live` (last seen within 2 days of September 30, so still open then: its lifespan is censored), `days_open`, `company_earlier` and the career-change flags.
- **Crawl artifacts** (batch first sightings, dropouts) are flagged and kept.

In [ ]:
# 2a Paths, the cleaned rows of the full file, and the cleaning report
CLEAN_DIR = DATA_DIR / "clean"
CLEAN_DIR.mkdir(exist_ok=True)
from src.listing_columns import first_country  # the first of a listing's or company's locations

LOAD_COLS = ["company_id", "first_seen", "last_seen", "posted", "status", "seniority", "language", "onet_code",
             "occupation_group", "occupation", "salary_low_usd", "salary_high_usd", "salary_unit", "has_desc", "has_html",
             "desc_words", "mentions_remote", "urgency", "years_required", "entry_wording", "changer_wording",
             "degree_required", "equivalent_experience", "n_locations", "country", "source_type", "platform"]
start = time.time()
stream = json.loads((FLAT_DIR / "stream_rows_progress.json").read_text())  # counts from the stream (no rows)
if not stream.get("finished"):
    raise RuntimeError("The rows stream hasn't finished: run `python -m src.stream_full --rows` first.")
jobs = load_rows(LOAD_COLS)
extras, labels = row_extras()  # same order as jobs: the index is each listing's row position in the rows table
jobs = pd.concat([jobs, extras], axis=1)
del extras
gc.collect()
print(f"loaded {len(jobs):,} cleaned rows × {jobs.shape[1]} columns in {time.time() - start:.0f}s, "
      f"{jobs.memory_usage(deep=True).sum() / 1e9:.1f} GB in memory")

companies = load_flat("companies", ["id", "attributes.location_data"])
companies["country"] = companies.pop("attributes.location_data").map(first_country)  # the company's first location (4b)
companies = companies.drop_duplicates("id")

report = []  # the cleaning report: one row per action, shown in 2f
def log(step, action, n):
    report.append({"step": step, "action": action, "rows": int(n)})
log("2a", "rows in the cleaned rows table (every English or unlabeled, US or no-country listing)", len(jobs))

In [ ]:
# 2b Standardize category labels (on each column's distinct labels, not row by row) and count the contract_types merges
for col in ["status", "seniority", "language", "salary_unit"]:
    before = jobs[col].nunique()
    jobs[col] = relabel(jobs[col], lambda c: str(c).strip().lower())
    log("2b", f"{col}: distinct labels merged by trimming and lowercasing", before - jobs[col].nunique())
before, after = labels["raw_contract_labels"], len(labels["contract_types"])  # unified part by part in 2a (row_extras)
log("2b", "contract_types: distinct labels merged (underscores, case, spacing)", before - after)

In [ ]:
# 2c Invalid dates, duplicate ids and reposts
DATA_END = pd.Timestamp("2026-09-30", tz="UTC")
END_OF_DATA = DATA_END + pd.Timedelta(days=1)

rules = {
    "missing first_seen or last_seen": jobs.first_seen.isna() | jobs.last_seen.isna(),
    "last_seen before first_seen": jobs.last_seen < jobs.first_seen,
    "dates after September 30 (the data's end date)": (jobs.first_seen > END_OF_DATA) | (jobs.last_seen > END_OF_DATA),
}
bad = pd.Series(False, index=jobs.index)
for name, mask in rules.items():
    log("2c", f"removed, {name}", (mask & ~bad).sum())  # each row once, under the first rule it fails
    bad |= mask
jobs = jobs[~bad]
del rules
gc.collect()

bad_posted = (jobs.posted > jobs.first_seen + pd.Timedelta(days=1)) | (jobs.posted < pd.Timestamp("2000-01-01", tz="UTC"))
jobs.loc[bad_posted, "posted"] = pd.NaT
log("2c", "posted blanked (after first_seen or before 2000)", bad_posted.sum())

In [ ]:
# 2c (cont.) Duplicate ids and reposts
# Duplicate ids and reposts both use 64-bit fingerprints built in 2a (id_fp; repost_fp = company, normalized title and
# description fingerprint): one number per listing instead of long text keys. Two different keys sharing a fingerprint is
# about a 1-in-200,000 chance across all 13 million listings.
dup_id = jobs.id_fp.duplicated()
log("2c", "removed, duplicate id", dup_id.sum())
jobs = jobs[~dup_id].drop(columns="id_fp")

jobs = jobs.sort_values("first_seen")
key = ["repost_fp"]
has_key = jobs.repost_fp.notna()
jobs["is_repost"] = has_key & jobs.duplicated(key)
log("2c", "flagged as repost (kept, excluded from the EDA tables)", jobs.is_repost.sum())
log("2c", "postings without a description, can't be checked for reposts", (~has_key).sum())

In [ ]:
# 2c (cont.) Repost features (earlier copies only, so they're known when a listing appears)
# Repost features: built only from earlier copies of the same job, so they're known when a listing appears
same_job = jobs[has_key].groupby(key)
jobs["times_listed"] = np.float32(np.nan)  # unknown without a description
jobs.loc[has_key, "times_listed"] = (same_job.cumcount() + 1).astype("float32")  # jobs are sorted by first_seen
jobs["days_since_first_listed"] = np.float32(np.nan)  # since the job's original listing
jobs.loc[has_key, "days_since_first_listed"] = (
    jobs.loc[has_key, "first_seen"] - same_job.first_seen.transform("min")).dt.total_seconds().div(86400).astype("float32")
jobs["days_since_repost_last_listed"] = np.float32(np.nan)  # since the job's previous listing; 0 for the first listing
jobs.loc[has_key, "days_since_repost_last_listed"] = (
    jobs.loc[has_key, "first_seen"] - same_job.first_seen.shift()).dt.total_seconds().fillna(0).div(86400).astype("float32")
del same_job
_ = gc.collect()

In [ ]:
# 2d Salary to one hourly rate: fix reversed ranges and wrong units, convert, blank outliers
PER_YEAR = {"hour": 2080, "day": 260, "week": 52, "month": 12, "year": 1}
HOURS_PER_YEAR = PER_YEAR["hour"]

def unit_factor(u):
    if pd.isna(u):
        return np.nan
    return next((f for k, f in PER_YEAR.items() if k in u), np.nan)

reversed_range = jobs.salary_low_usd > jobs.salary_high_usd
jobs.loc[reversed_range, ["salary_low_usd", "salary_high_usd"]] = (
    jobs.loc[reversed_range, ["salary_high_usd", "salary_low_usd"]].to_numpy())
mid = jobs[["salary_low_usd", "salary_high_usd"]].mean(axis=1)
log("2d", "salary present (midpoint of low and high)", mid.notna().sum())
log("2d", "salary range reversed (low above high): swapped", reversed_range.sum())

yearly_but_hourly = cat_contains(jobs.salary_unit, "year") & mid.between(7, 200)              # "$25 a year" is hourly pay
hourly_but_yearly = cat_contains(jobs.salary_unit, "hour") & mid.between(10_000, 1_000_000)   # "$85,000 an hour" is yearly pay
factor = cat_map(jobs.salary_unit, unit_factor).mask(yearly_but_hourly, PER_YEAR["hour"]).mask(hourly_but_yearly, PER_YEAR["year"])
log("2d", "labelled yearly but clearly hourly ($7-$200): re-read as hourly", yearly_but_hourly.sum())
log("2d", "labelled hourly but clearly yearly ($10k-$1M): re-read as yearly", hourly_but_yearly.sum())
log("2d", "salary present but unit missing or not recognized", (mid.notna() & factor.isna()).sum())

hourly = mid * factor / HOURS_PER_YEAR
outlier = ~hourly.between(10_000 / HOURS_PER_YEAR, 1_000_000 / HOURS_PER_YEAR) & hourly.notna()
jobs["salary_hourly_usd"] = hourly.mask(outlier).astype("float32")
jobs["pay_suspect"] = reversed_range | yearly_but_hourly | hourly_but_yearly
log("2d", "hourly pay outside $4.81-$480.77 ($10k-$1M a year), blanked", outlier.sum())
log("2d", "usable hourly pay", jobs.salary_hourly_usd.notna().sum())

In [ ]:
# 2e Derived timing columns, career-change flags, company history and crawl artifacts; the analysis table (originals only)
LIVE_DAYS = 2
jobs["live"] = (DATA_END - jobs.last_seen).dt.days <= LIVE_DAYS
jobs["days_open"] = ((jobs.last_seen - jobs.first_seen).dt.total_seconds() / 86400).astype("float32")
jobs["month"] = jobs.first_seen.dt.tz_localize(None).dt.to_period("M")
jobs["occupation_group"] = with_missing(jobs.occupation_group)

# Career-change accessibility flags (see the decisions in section 2)
ENTRY_LEVELS = {"junior"}  # the only entry-level label (the others: mid_senior, manager, head, director, vice_president, president, c_level, partner, founder, not_set)
jobs["entry_seniority"] = jobs.seniority.isin(ENTRY_LEVELS)
jobs["low_experience_ask"] = jobs.years_required <= 2  # False where no years are stated
jobs["career_open"] = jobs.entry_seniority | jobs.low_experience_ask | jobs.entry_wording | jobs.changer_wording

# Company history in any field, counted before the cut (company volume is a confounder for every later notebook):
# each listing's count of its company's listings first seen before it (same-day listings count together; reposts included)
jobs["company_earlier"] = (jobs.groupby("company_id", observed=True).first_seen.rank(method="min") - 1).astype("float32")

In [ ]:
# 2e (cont.) Crawl artifacts: the one-day batch rule
# Crawl artifacts (flagged, kept; definitions shared with the hackathon app's signals):
# - batch_first_seen: the company has 5+ listings and 60%+ of them were first seen on one day (the company was added
#   to the crawl, or its site rebuilt), and this listing is one of them. The data's first two days are left out: every
#   listing already up then is first seen there (they're flagged partial_month instead).
# - crawl_dropout: 5+ of the company's listings disappeared (not live) and 60%+ of those were last seen on one day
#   (the crawler likely lost the site), and this listing is one of them: its days_open may end early.
BATCH_MIN, BATCH_SHARE = 5, 0.6
def one_day_batch(company_codes, day, eligible):
    """True for listings on the day holding BATCH_SHARE+ of their company's eligible listings (BATCH_MIN+ of them)."""
    d = pd.DataFrame({"co": company_codes[eligible], "day": day[eligible]})
    d = d[d.co >= 0]
    per_day = d.groupby(["co", "day"]).size()
    per_co = per_day.groupby(level="co").sum()
    top = per_day.sort_values(ascending=False).groupby(level="co").head(1)  # the busiest day per company
    top = top[(per_co.reindex(top.index.get_level_values("co")).to_numpy() >= BATCH_MIN)
              & (top.to_numpy() / per_co.reindex(top.index.get_level_values("co")).to_numpy() >= BATCH_SHARE)]
    hit = pd.MultiIndex.from_arrays([company_codes, day]).isin(top.index)
    return pd.Series(hit & eligible, index=jobs.index)

In [ ]:
# 2e (cont.) Applying the batch and dropout rules
codes = jobs.company_id.cat.codes.to_numpy()
first_day = (jobs.first_seen.astype("int64") // 86_400_000_000_000).to_numpy()
last_day = (jobs.last_seen.astype("int64") // 86_400_000_000_000).to_numpy()
jobs["batch_first_seen"] = one_day_batch(codes, first_day, (jobs.first_seen >= pd.Timestamp("2026-04-01", tz="UTC")).to_numpy())
jobs["crawl_dropout"] = one_day_batch(codes, last_day, (~jobs.live).to_numpy())
artifact_stats = {
    "listings first seen in a company's one-day batch": jobs.batch_first_seen.mean(),
    "companies with such a batch (of companies with 5+ listings)": (
        jobs.loc[jobs.batch_first_seen, "company_id"].nunique() / max((jobs.company_id.value_counts() >= BATCH_MIN).sum(), 1)),
    "listings that disappeared in a company's one-day dropout": jobs.crawl_dropout.mean(),
    "companies with such a dropout (of companies with 5+ listings)": (
        jobs.loc[jobs.crawl_dropout, "company_id"].nunique() / max((jobs.company_id.value_counts() >= BATCH_MIN).sum(), 1)),
}
del codes, first_day, last_day

In [ ]:
# 2e (cont.) To disk, and the analysis table (originals only)
# Memory: every cleaned listing (reposts flagged) goes to disk for section 4; jobs keeps only the columns the
# repost and file-order checks below need, and jo (originals) is read back from disk.
CLEANED_PATH = CLEAN_DIR / "cleaned_rows_full.parquet"
jobs.to_parquet(CLEANED_PATH)
JOBS_COLS = ["company_id", "first_seen", "occupation_group", "onet_code", "is_repost", "language", "country", "has_desc",
             "mentions_remote", "contract_bits"]
jobs = jobs[JOBS_COLS]
gc.collect()
jo = pd.read_parquet(CLEANED_PATH, filters=[("is_repost", "==", False)])
log("2e", "postings in the analysis table (originals only)", len(jo))

In [ ]:
# 2f The cleaning report, the analysis table at a glance and the crawl artifacts
show("Cleaning report (every action, in order)", pd.DataFrame(report).assign(
    **{"share of start": lambda d: (d.rows / report[0]["rows"]).round(4)}))
show("Analysis table at a glance (originals: the rows for the every-field comparison)", pd.DataFrame({"value": {
    "postings": compact(len(jo)),
    "companies": compact(jo.company_id.nunique()),
    "occupation groups": jo.occupation_group.nunique(),
    "still open on September 30": f"{jo.live.mean():.1%}",
    "first_seen from": jo.first_seen.min().date(),
    "first_seen to": jo.first_seen.max().date(),
}}))
show("Crawl artifacts (flagged, kept): every cleaned listing", pd.Series(artifact_stats).to_frame("share"))
section_done("2. Cleaning (every listing)")

**What the data shows**
- **Cleaning removed no listings:** no duplicate ids, no impossible timelines, no reversed pay ranges or mislabelled units. It flagged 1.4M reposts (10%; 17% of listings can't be checked, having no description), blanked 349K employer posting dates (2.7%: placeholders before 2000, or dates after the crawler first saw the listing) and 984 pay values outside $10K–$1M a year.
- **11.7M original listings from 324K companies** feed the every-field comparison (section 3); 22.3% were still open on September 30.
- **Crawl artifacts are small but concentrated:** 2.4% of listings were first seen in a company's one-day batch (17% of companies with 5+ listings have one), and 1.2% disappeared in a one-day dropout (9.9% of those companies). They distort single companies more than the totals, so they're flagged, never dropped.

## 3. Q1: How do computer and mathematical roles compare with every other field?

Every occupation group before the scope cut, on original English listings; weekly rates use full weeks only. **Context, not cause:** fields differ in company size, source and seniority.

In [ ]:
# 3a Shared constants; full weeks for weekly rates
COMPANY = "company_id"
TOP = 15
SCOPE_GROUP = "Computer and Mathematical"
# Weekly rates use full weeks only: the first week holds the crawler's start-up backlog (listings already up when it
# began), and the last is cut off by the data's end date (Wednesday, September 30). The listings stay; only per-week figures
# leave those two weeks out.
week = jo.first_seen.dt.tz_localize(None).dt.to_period("W")
FIRST_WEEK, LAST_WEEK = week.min(), week.max()
jo["full_week"] = (week > FIRST_WEEK) & (week < LAST_WEEK)
FULL_WEEKS = week[jo.full_week].nunique()
del week
in_market = jo.language.eq("en").to_numpy()  # section 4's language rule
print(f"full weeks: {FULL_WEEKS} (first week {FIRST_WEEK}, last week {LAST_WEEK} left out of weekly rates)")

In [ ]:
# 3b Every field side by side (original listings, English; text measures among listings with a description)
FIELD_COLS = ["occupation_group", COMPANY, "days_open", "live", "salary_hourly_usd", "entry_seniority", "career_open",
              "years_required", "degree_required", "mentions_remote", "has_desc", "full_week"]
market = jo.loc[in_market, FIELD_COLS]

def field_stats(g):
    curve = km_curve(g.days_open, ~g.live)
    d = g[g.has_desc]
    stated = g.years_required.dropna()
    return pd.Series({
        "listings": len(g),
        "share of all listings": len(g) / len(market),
        "new per week": g.full_week.sum() / FULL_WEEKS,
        "companies": g[COMPANY].nunique(),
        "median hourly pay (USD)": g.salary_hourly_usd.median(),
        "pay disclosed": g.salary_hourly_usd.notna().mean(),
        "median days open (Kaplan–Meier)": curve[curve <= 0.5].index.min(),
        "closed within 7 days": 1 - surv_at(curve, 7),
        "entry-level seniority": g.entry_seniority.mean(),
        "asks 2 years or fewer (where stated)": (stated <= 2).mean(),
        "open to career changers (any sign)": g.career_open.mean(),
        "degree required": d.degree_required.mean(),
        "mentions remote": d.mentions_remote.mean(),
    })

with warnings.catch_warnings():  # a field with no disclosed pay gives an empty median: NaN, as intended
    warnings.simplefilter("ignore", RuntimeWarning)
    fields = market.groupby("occupation_group", observed=True).apply(field_stats, include_groups=False)
fields = fields.drop(index="(missing)", errors="ignore").sort_values("listings", ascending=False)
show_shaded(f"3a. Every field side by side ({len(fields)} occupation groups; original English listings; each column shaded on its own scale)",
            fields, highlight=SCOPE_GROUP)

measures = fields.drop(columns="scope", errors="ignore")  # 3b builds no scope column; the group is highlighted instead
if SCOPE_GROUP in measures.index:
    show(f"3a. Where computer and mathematical stands among the {len(measures)} fields", pd.DataFrame({
        "computer and mathematical": measures.loc[SCOPE_GROUP],
        "median field": measures.median(),
        "rank (1 = highest)": measures.rank(ascending=False, method="min").loc[SCOPE_GROUP].astype(int),
    }), by="rows")

In [ ]:
# 3c chart: openness to career changers vs. pay, every field (bubble size = listings; computer and mathematical highlighted)
pts = fields
x, y = 100 * pts["open to career changers (any sign)"], pts["median hourly pay (USD)"]
fig, ax = plt.subplots(figsize=(10, 5.6))
ax.scatter(x, y, s=40 + 900 * pts.listings / pts.listings.max(), alpha=0.8, edgecolor=SURFACE,
           color=[ORANGE if n == SCOPE_GROUP else BLUE for n in pts.index])
for name in pts.index[:12].union([SCOPE_GROUP]).intersection(pts.index):
    ax.annotate(str(name)[:30], (x[name], y[name]), xytext=(6, 4), textcoords="offset points", fontsize=8,
                color=INK if name == SCOPE_GROUP else INK_2, weight="bold" if name == SCOPE_GROUP else "normal")
ax.axvline(100 * market.career_open.mean(), color=MUTED, ls="--", lw=1)
ax.axhline(market.salary_hourly_usd.median(), color=MUTED, ls="--", lw=1)
ax.set(title="Open to career changers vs. pay, every field (dashed: all listings; orange: computer and mathematical)",
       xlabel="% of listings with any sign of being open to career changers", ylabel="median hourly pay ($)")
finish()

**What the data shows**
- **Computer and mathematical is the best-paid and most remote field:** a median **$66.73 an hour** vs. $26.44 for the median field, and 33% of listings mention remote work vs. 9.5%. It's the 4th largest (8.1% of English listings, about 34K new a week).
- **It's also among the hardest to enter:** 9.4% entry level (22nd of 23 fields), **17% of listings that state years ask for 2 or fewer (last of 23; median field 52%)**, and 21% show any sign of openness (22nd). The most open fields (food preparation, building and grounds, transportation) pay $18–22 an hour.
- **Its listings close at an average pace** (median 23.4 days, 14th), though 30% are gone within a week (4th). Worth the move, hard to enter: exactly where hiding what a newcomer can't get, and ranking the rest, helps.

**Saved for later notebooks:** the occupation summary (pay, timing and openness per occupation; the compass's "your pay increase"), `occupation_summary.pkl`, and company activity (each company's listings in any field and newest listing; the "company still posting" ghost signal), `company_activity.parquet`.

In [ ]:
# 3d Occupation summary (every field; aggregates only, not shown here) and company activity (every field)
def occ_stats(g):
    return pd.Series({
        "listings": len(g),
        "companies": g[COMPANY].nunique(),
        "pay disclosed": g.salary_hourly_usd.notna().mean(),
        "median hourly pay (USD)": g.salary_hourly_usd.median(),
        "median days open (Kaplan–Meier)": km_median(g.days_open, ~g.live),
        "entry-level seniority": g.entry_seniority.mean(),
        "open to career changers (any sign)": g.career_open.mean(),
    })

with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    occupation_summary = (jo.loc[in_market, ["occupation_group", "occupation", COMPANY, "salary_hourly_usd", "days_open",
                                             "live", "entry_seniority", "career_open"]]
                          .groupby(["occupation_group", "occupation"], observed=True).apply(occ_stats, include_groups=False))
OCC_SUMMARY_PATH = CLEAN_DIR / "occupation_summary.pkl"
occupation_summary.to_pickle(OCC_SUMMARY_PATH)

company_activity = jobs.groupby(COMPANY, observed=True).agg(listings_any_field=("first_seen", "size"),
                                                            newest_listing_any_field=("first_seen", "max"))
COMPANY_ACTIVITY_PATH = CLEAN_DIR / "company_activity.parquet"
company_activity.to_parquet(COMPANY_ACTIVITY_PATH)
del market
gc.collect()
print(f"saved {compact(len(occupation_summary))} occupations in {occupation_summary.index.get_level_values(0).nunique()} groups "
      f"({OCC_SUMMARY_PATH.name}) and {compact(len(company_activity))} companies ({COMPANY_ACTIVITY_PATH.name})")
section_done("3. Every field compared, summaries saved")

## 4. The scope: where the data narrows to the roles TripleTen trains for

Before the cut, every field; after it, the scope only. Nothing is deleted: the cleaned rows stay on disk.

| Decision | Choice | Why |
|:--|:--|:--|
| **Language** | English; listings with no language label dropped | The text features and the app are English; unlabeled listings have no description (4b) |
| **Country** | Listings that name the US; no-country listings dropped | Pay is in dollars and the job-growth projections (BLS) are for the US; only these are known US openings (4b) |
| **Scope** | The roles TripleTen trains for, plus adjacent tech roles; credentialed math out | TripleTen's program catalog, a neutral definition (4a) |
| **Time** | April–September; late March kept, flagged `partial_month` | Left out of per-month counts only |

Seniority, long-open listings and staffing agencies are labelled or hidden, never dropped.

### 4a Why this scope

- **TripleTen's program catalog** (tripleten.com, checked 2026-10-10) defines it: a neutral, outside definition. IT support, networks and systems, database and IT project management have no program but stay as **adjacent tech roles,** since IT support is the most open entry point.
- **The data supports it:** a switch pays off (section 3), 2,813 career changers in the résumé data moved into these roles (`Profiles.ipynb`), and they're hard to enter, which is where hiding and ranking help.

### 4b Language and country

What the two rules cost: no-country listings may be US remote jobs, and unlabeled listings may be the ones with no description.

In [ ]:
# 4b Checks: what the country and language rules cost (every cleaned listing, every field)
REMOTE_BITS = sum(1 << i for i, label in enumerate(labels["contract_types"][:63]) if "remote" in label)  # "fully remote", "100% remote"...
remote_contract = pd.Series((jobs.contract_bits.to_numpy() & REMOTE_BITS) != 0, index=jobs.index)
show("4b. Remote mentions, by whether the country is known", pd.DataFrame({
    "country_known": jobs.country.notna(), "mentions_remote": jobs.mentions_remote, "remote_contract_type": remote_contract,
}).groupby("country_known").agg(
    listings=("mentions_remote", "size"),
    mentions_remote=("mentions_remote", "mean"),
    remote_contract_type=("remote_contract_type", "mean"),
))
co_country = cat_map(jobs[COMPANY], companies.set_index("id").country.eq("United States").astype(float))
no_country = jobs.country.isna()
show("4b. Listings with no country: where their companies are based", pd.DataFrame({"share of no-country listings": {
    "company based in the US": (no_country & co_country.eq(1)).sum() / max(no_country.sum(), 1),
    "company based elsewhere (dropped)": (no_country & co_country.eq(0) & jobs[COMPANY].map(
        companies.set_index("id").country).notna().to_numpy()).sum() / max(no_country.sum(), 1),
    "company location unknown or not in the companies table (dropped)": (no_country & ~co_country.eq(1) & jobs[COMPANY].map(
        companies.set_index("id").country).isna().to_numpy()).sum() / max(no_country.sum(), 1),
}}))
show("4b. Language label vs. description (listings)",
     pd.crosstab(jobs.language.notna().map({True: "language known", False: "no language label"}),
                 jobs.has_desc.map({True: "has description", False: "no description"}), margins=True))
print(f"listings with a country other than the US after the stream: {(jobs.country.notna() & jobs.country.ne('United States')).mean():.2%}")
del remote_contract, co_country, no_country
gc.collect()

**What the data shows**
- **Listings with no country (8.0% of cleaned listings) are largely US remote jobs:** 30% mention remote work vs. 13% with a country, and they use a remote contract label more often (0.83% vs. 0.51%). 93% come from a US-based company, 2.6% from a company elsewhere and 4.3% from an unknown one; all are dropped.
- **No language label means no description:** 623K of the 624K unlabeled listings have none, so dropping them also removes about 28% of all listings without a description (623K of 2.2M). 95% of cleaned listings are English, and no listing outside the US remains after the stream.

### 4c The scope cut

In the scope: occupation group *Computer and Mathematical* or an O*NET code starting 15-, or one of two TripleTen roles outside the group (sales engineers, market research analysts); minus credentialed math (actuaries, mathematicians). Each listing gets a **role bucket** and a **TripleTen program** (`src/career_switch.py`), or "adjacent tech role" where no program trains for it.

In [ ]:
# 4c The scope filter (the cut): roles TripleTen trains for, plus adjacent tech roles
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.dataset as ds
from src.career_switch import EXTRA_ORDER, PROGRAM_ORDER, ROLE_ORDER, role_bucket, tripleten_program

FILTERS = {"language": "en", "country": "United States", "first_month": "2026-04"}
english = jobs.language.eq(FILTERS["language"])
english_us = english & jobs.country.eq(FILTERS["country"])
by_group = jobs.occupation_group.eq(SCOPE_GROUP)
by_code = cat_contains(jobs.onet_code, r"^15-")
bucket = relabel(jobs.onet_code, role_bucket)  # role bucket from the O*NET code (None outside the mapped codes)
added = bucket.isin(EXTRA_ORDER)  # the two TripleTen roles outside the group (EXTRA_ONET_BUCKETS)
credentialed = bucket.eq("credentialed math")
in_scope = english_us & (by_group | by_code | added) & ~credentialed
show("4c. How English US listings enter the scope (share of English US listings)", pd.DataFrame({"share of English US listings": {
    "occupation group is computer and mathematical": (english_us & by_group).mean() / english_us.mean(),
    "  of which the O*NET code also starts with 15-": (english_us & by_group & by_code).mean() / english_us.mean(),
    "only the O*NET code starts with 15- (group missing or different)": (english_us & by_code & ~by_group).mean() / english_us.mean(),
    "credentialed math (actuaries, mathematicians): set aside": (english_us & (by_group | by_code) & credentialed).mean() / english_us.mean(),
    "a TripleTen role outside the group (sales engineers, business and market research analysts)": (english_us & added & ~(by_group | by_code)).mean() / english_us.mean(),
}}))

In [ ]:
# 4c (cont.) The funnel, and the scope's rows read back from disk
steps = [("listings in the full file", stream["records"]),
         ("English or unlabeled, US or no country (the stream, 2a)", stream["rows"]),
         ("cleaned: impossible dates and duplicate ids removed (2c)", len(jobs)),
         ("English only", english.sum()),
         ("United States only: the listing names the US", english_us.sum()),
         ("TripleTen roles plus adjacent tech roles (computer and mathematical, or an added TripleTen role)", (english_us & (by_group | by_code | added)).sum()),
         ("credentialed math out (actuaries, mathematicians)", in_scope.sum())]
scope_rows = jobs.index[in_scope.to_numpy()].to_numpy()

# Only the scope's rows are read back from disk (every column); every-field tables are no longer needed
del jo, jobs, english, english_us, by_group, by_code, bucket, added, credentialed, in_scope, in_market
gc.collect()
final = ds.dataset(CLEANED_PATH).to_table(filter=pc.field("__index_level_0__").isin(pa.array(scope_rows))).to_pandas()
final["partial_month"] = final.month < pd.Period(FILTERS["first_month"], "M")  # hide, never remove
week = final.first_seen.dt.tz_localize(None).dt.to_period("W")
final["full_week"] = (week > FIRST_WEEK) & (week < LAST_WEEK)
final["role_bucket"] = with_missing(relabel(final.onet_code, role_bucket), "(unmapped 15- code)")
final["tripleten_program"] = with_missing(relabel(final.onet_code, tripleten_program), "adjacent tech role")
steps.append((f"first seen before {FILTERS['first_month']}: kept, flagged partial_month ({final.partial_month.mean():.1%})", len(final)))

In [ ]:
# 4c (cont.) The funnel table, and the scope by role bucket and TripleTen program
funnel = pd.DataFrame(steps, columns=["step", "listings"]).set_index("step")
funnel["removed"] = (-funnel.listings.diff()).fillna(0).astype(int)
funnel["share of the file"] = funnel.listings / funnel.listings.iloc[0]
funnel["share removed at this step"] = funnel.removed / funnel.listings.shift().fillna(funnel.listings.iloc[0])
show("4c. From the full file to the scope (every step)", funnel)
buckets = final.role_bucket.value_counts(normalize=True)
show("4c. The scope by role bucket", pd.DataFrame({"share of the scope": buckets[buckets > 0]}))
programs = final.tripleten_program.value_counts(normalize=True).reindex(PROGRAM_ORDER).dropna()
show("4c. The scope by TripleTen program (adjacent tech role = no program trains for it)",
     pd.DataFrame({"share of the scope": programs[programs > 0]}))
unmapped = final.role_bucket.eq("(unmapped 15- code)")
codes = final.onet_code[unmapped].value_counts(normalize=True)
print(f"scope listings whose O*NET code maps to no role bucket: {unmapped.mean():.2%}")
if unmapped.any():
    show("4c. Unmapped O*NET codes in the scope (add them to src/career_switch.py if they're large)",
         codes[codes > 0].head(10).to_frame("share of unmapped listings"))

**What the data shows**
- **The scope is 908K listings, 6.8% of the file:** computer and mathematical is 7.2% of English US listings, the two added TripleTen roles add 0.69%, and credentialed math is 0.05%. The O*NET-code fallback adds nothing (the group is always present), and every scope code maps to a role bucket.
- **Software development is the largest bucket (29%),** then networks and systems (19%), IT project management (12%) and data science (12%); technical sales is the smallest (1.1%).
- **By TripleTen program:** AI Software Engineering 28%, Data Analytics 14%, Cybersecurity 5.0%, AI & Machine Learning 4.8%, Quality Assurance 3.7%, UX/UI Design 1.1%, Forward Deployed Engineering / AI Automation 1.1%; **42% are adjacent tech roles.**

### 4d Fields only the scope carries

From the stream's full records: the **full description** (`scope_descriptions.parquet`, for EDA_SDA_3), an **anonymous recruiter key** and title (names and contacts aren't kept), **pay as written** (parsed into hourly pay where missing), **location as written** and the **url**.

In [ ]:
# 4d Scope-only fields from the stream's scope parts (every original field); descriptions written to disk
from src.rows_table import ids_at, scope_fields
from src.listing_columns import pay_from_text
start = time.time()
DESC_PATH = CLEAN_DIR / "scope_descriptions.parquet"
ids = ids_at(final.index)
extras = scope_fields(ids, DESC_PATH)
final = final.join(extras)
del extras
gc.collect()

# Pay from the text where salary_data had none (same conversion and bounds as 2d)
needs = final.salary_hourly_usd.isna() & final.salary_text.notna()
parsed = final.loc[needs, "salary_text"].map(pay_from_text).dropna()
low, high, unit = (parsed.str[k] for k in range(3))
hourly = (low + high) / 2 * unit.map(PER_YEAR) / HOURS_PER_YEAR
hourly = hourly[hourly.between(10_000 / HOURS_PER_YEAR, 1_000_000 / HOURS_PER_YEAR)]
final["pay_from_text"] = False
final.loc[hourly.index, "salary_hourly_usd"] = hourly.astype("float32")
final.loc[hourly.index, "pay_from_text"] = True
missing = final.record_type.isna()
if missing.mean() > 0.01:
    print(f"NOTE: {missing.mean():.1%} of scope listings aren't in the stream's saved full records. If they're the two added TripleTen "
          "roles, run `python -m src.stream_full --extra` (about an hour), then rerun from 4d.")
show("4d. Scope-only fields: share of scope listings with each", pd.DataFrame({"share": {
    "found in the stream's scope parts": final.record_type.notna().mean(),
    "pay as written": final.salary_text.notna().mean(),
    "location as written": final.location_text.notna().mean(),
    "hourly pay before parsing the text": final.salary_hourly_usd.notna().mean() - final.pay_from_text.mean(),
    "hourly pay added from the text": final.pay_from_text.mean(),
}}).round(4))
print(f"descriptions saved to {DESC_PATH.name} ({DESC_PATH.stat().st_size / 1e9:.2f} GB) in {time.time() - start:.0f}s")
section_done("4. Filters and scope-only fields")

**What the data shows**
- **Every scope listing was found in the stream's full records;** the descriptions are saved for EDA_SDA_3 (2.3 GB of text).
- **Pay as written exists for 45%,** and parsing it added hourly pay for 1.4% of listings (43% to 45%). 90% carry a location as written.

### 4e The scope table at a glance

In [ ]:
# 4e The scope at a glance, how complete the key fields are, and what the other fields hold
show("4e. The scope table", pd.DataFrame({"value": {
    "listings": compact(len(final)),
    "companies": compact(final.company_id.nunique()),
    "role buckets": final.role_bucket.nunique(),
    "full months": final.loc[~final.partial_month, "month"].nunique(),
    "late-March listings (partial_month)": f"{final.partial_month.mean():.1%}",
}}))
KEY_FIELDS = {
    "first and last seen": final.first_seen.notna() & final.last_seen.notna(),
    "role bucket": final.role_bucket.ne("(unmapped 15- code)"),
    "source": final.source_type.notna() & final.source_type.ne("(missing)"),
    "description": final.has_desc.fillna(False).astype(bool),
    "seniority": final.seniority.notna() & final.seniority.ne("not_set"),
    "hourly pay": final.salary_hourly_usd.notna(),
    "employer's posting date": final.posted.notna(),
    "years of experience stated": final.years_required.notna(),
}
show("4e. Share of scope listings with each key field", pd.DataFrame(KEY_FIELDS).mean().round(3).to_frame("share filled"))

def label_counts(bits, names, n=TOP):
    return (bit_counts(bits, names) / len(final)).head(n).round(3).to_frame("share")

contract = label_counts(final.contract_bits, labels["contract_types"])
contract.loc["(none)"] = round((final.contract_bits == 0).mean(), 3)
show("4e. contract_types (a listing can have several)", contract)
print(f"scope listings whose company is in the companies table: {pct(final[COMPANY].isin(companies['id']).mean())}")

**What the data shows**
- **The scope table is 908K listings from 72K companies.** Timing, role bucket, source and seniority are complete; description 78%, pay 45%, years stated 43%, employer posting date 39%.
- **Contract types are messy** (161 labels; full time 55%, hybrid 20%, contract 8.8%, none 27%). Every scope company is in the companies table.

## 5. Q2: Which factors should drive prioritization?

**Scope listings only** (originals, `so`; repost rates use every listing). A factor only helps ranking if it **varies**: 5a shows the main factors by role bucket and program, with each one's spread (max ÷ min); 5b–5h look closer at one family each.

In [ ]:
# 5.0 The scope's original listings, and the role buckets in display order
so = final[~final.is_repost]
BUCKETS = [b for b in ROLE_ORDER + ["other computer roles"] + EXTRA_ORDER + ["(unmapped 15- code)"] if (so.role_bucket == b).any()]
PROGRAMS = [p for p in PROGRAM_ORDER if (so.tripleten_program == p).any()]
print(f"scope: {compact(len(final))} listings ({compact(len(so))} originals) in {len(BUCKETS)} role buckets")

### 5a The main factors, by role bucket and program

In [ ]:
# 5a The main factors for one group (g = originals)
def targets(g):
    curve = km_curve(g.days_open, ~g.live)
    return pd.Series({
        "median days open (Kaplan–Meier)": curve[curve <= 0.5].index.min(),
        "new postings per week": g.full_week.sum() / FULL_WEEKS,
        "salary disclosed": g.salary_hourly_usd.notna().mean(),
        "median hourly pay (USD)": g.salary_hourly_usd.median(),
        "mentions remote": g.mentions_remote.mean(),
        "median words per description": g.desc_words.where(g.has_desc.fillna(False).astype(bool)).median(),
        "manager-level seniority": (g.seniority == "manager").mean(),
        "entry-level seniority": g.entry_seniority.mean(),
        "open to career changers (any sign)": g.career_open.mean(),
    })

by_bucket = pd.DataFrame({b: targets(so[so.role_bucket == b]) for b in BUCKETS}).T
overview = pd.DataFrame({
    "whole scope": targets(so),
    "bucket min": by_bucket.min(),
    "bucket median": by_bucket.median(),
    "bucket max": by_bucket.max(),
})
overview["spread (max / min)"] = overview["bucket max"] / overview["bucket min"].replace(0, np.nan)
show("5a. The main factors: the whole scope, and the range across role buckets", overview, by="rows")
show_shaded("5a. The main factors by role bucket (each column shaded on its own scale)", by_bucket)
by_program = pd.DataFrame({p: targets(so[so.tripleten_program == p]) for p in PROGRAMS}).T
show_shaded("5a. The same factors by TripleTen program (adjacent tech role = no program trains for it)", by_program,
            highlight="adjacent tech role")

In [ ]:
# 5a chart: how much each factor differs between role buckets
fig, ax = plt.subplots(figsize=(8, 4))
bar_h(ax, overview["spread (max / min)"], "How much each factor differs between role buckets",
      "spread: highest bucket ÷ lowest bucket", "{:.1f}×")
ax.axvline(1, color=INK_2, lw=1)
finish()

**What the data shows**
- **Accessibility varies most between buckets:** entry-level seniority 10.8×, open to career changers 4.8×.
- **Pay varies 2.4×** ($31.10 an hour in IT support to $73.54 in software development) **and volume 27×** (340 new a week in technical sales to 9,200 in software development), while timing varies only 1.7× (medians of 22–38 days).
- **By program:** Data Analytics roles are the most open (30% show any sign) at $47.84 an hour; AI Software Engineering and AI & Machine Learning pay the most (about $74) but are among the least open (18–19%).

### 5b Application window

Share still open N days after first seen (Kaplan–Meier, so still-open listings count as unfinished): the basis for an **apply-by estimate**.

In [ ]:
# 5b Application window by role bucket
DAYS = [3, 7, 14, 30, 60]
window = {b: km_survival(so.days_open[so.role_bucket == b], ~so.live[so.role_bucket == b], DAYS) for b in BUCKETS}
window["(whole scope)"] = km_survival(so.days_open, ~so.live, DAYS)
show_shaded("5b. Share of listings still open N days after first seen (Kaplan–Meier)",
            pd.DataFrame(window, index=[f"open after {d}d" for d in DAYS]).T, highlight="(whole scope)")

In [ ]:
# 5b chart: survival curves for the fastest and slowest role buckets
grid = np.arange(0, 121)
def on_grid(curve):
    return curve.reindex(curve.index.union(grid)).ffill().reindex(grid).fillna(1.0)

medians = {b: km_median(so.days_open[so.role_bucket == b], ~so.live[so.role_bucket == b]) for b in BUCKETS}
medians = {b: m for b, m in medians.items() if pd.notna(m)}
fast, slow = min(medians, key=medians.get), max(medians, key=medians.get)
fig, ax = plt.subplots(figsize=(9.5, 4.2))
for b in BUCKETS:
    if b not in (fast, slow):
        g = so[so.role_bucket == b]
        ax.step(grid, on_grid(km_curve(g.days_open, ~g.live)), where="post", color=MUTED, lw=1)
ax.plot([], [], color=MUTED, lw=1, label="other role buckets")
for b, color, tag in ((fast, ORANGE, "fastest"), (slow, BLUE, "slowest")):
    g = so[so.role_bucket == b]
    ax.step(grid, on_grid(km_curve(g.days_open, ~g.live)), where="post", color=color, label=f"{b} ({tag})")
ax.step(grid, on_grid(km_curve(so.days_open, ~so.live)), where="post", color=INK, lw=1.5, ls="--", label="whole scope")
ax.set(title="Share of listings still open, by days since first seen", xlabel="days since first seen",
       ylabel="share still open", xlim=(0, 120), ylim=(0, 1.02))
ax.legend(fontsize=8.5)
finish()

In [ ]:
# 5b check: timing with and without crawl artifacts (scope originals; flags from 2e)
views = {"every listing": pd.Series(True, index=so.index),
         "without batch first sightings": ~so.batch_first_seen,
         "without crawl dropouts": ~so.crawl_dropout,
         "without either": ~so.batch_first_seen & ~so.crawl_dropout}
show("5b check: timing with and without crawl artifacts", pd.DataFrame({name: {
    "share of listings": m.mean(),
    "median days open (Kaplan–Meier)": km_median(so.days_open[m], ~so.live[m]),
    "open after 7d": km_survival(so.days_open[m], ~so.live[m], [7])[0],
    "open after 30d": km_survival(so.days_open[m], ~so.live[m], [30])[0],
} for name, m in views.items()}).T)

**What the data shows**
- **The window is short:** 29% of scope listings close within a week, the median is 24.2 days, and 45% are still open after 30 days.
- **Most buckets move alike** (67–74% still open after a week). **Technical sales is the slow outlier** (84% after a week, median 38 days); IT support, database, quality assurance and other computer roles are the fastest (median 22 days).
- **Crawl artifacts barely move timing:** without them the median falls from 24.2 to 23.1 days, and the share open after a week stays at 71%.

### 5c Competition: company hiring volume

Volume is the company's listings **in any field first seen before each listing** (`company_earlier`), as the model will see it.

In [ ]:
# 5c Company concentration by role bucket (scope listings)
def concentration(g):
    per_co = g[COMPANY].value_counts()
    per_co = per_co[per_co > 0]  # company_id is a category: drop companies with no listings here
    return pd.Series({
        "companies": per_co.size,
        "postings per company (median)": per_co.median(),
        "top 1% of companies' share": per_co.head(max(1, per_co.size // 100)).sum() / len(g),
        "top 10% of companies' share": per_co.head(max(1, per_co.size // 10)).sum() / len(g),
        "companies with 5+ postings": (per_co >= 5).sum(),
    })
show_shaded("5c. Company concentration by role bucket",
            so[[COMPANY, "role_bucket"]].groupby("role_bucket", observed=True).apply(concentration, include_groups=False).reindex(BUCKETS))

In [ ]:
# 5c Do high-volume companies' listings close faster or slower? (earlier listings in any field: company_earlier, 2e)
# Company-volume bands (listings in any field)
BANDS = ["1-4", "5-49", "50-499", "500+"]
def size_band(n):
    return pd.cut(n.fillna(1), [0, 4, 49, 499, np.inf], labels=BANDS)
asof_band = size_band(so.company_earlier + 1)
since_may = so.first_seen >= pd.Timestamp("2026-05-01", tz="UTC")  # a month or more of history behind each

def band_medians(mask):
    return so.loc[mask, ["days_open", "live"]].groupby(asof_band[mask], observed=True).apply(lambda g: km_median(g.days_open, ~g.live))

show("5c. Median days open (Kaplan–Meier) by company volume (earlier listings in any field)", pd.DataFrame({
    "share of listings": asof_band.value_counts(normalize=True),
    "median days open (Kaplan–Meier)": band_medians(pd.Series(True, index=so.index)),
    "median days, first seen May or later": band_medians(since_may),
    "repost rate": final.is_repost.groupby(size_band(final.company_earlier + 1), observed=True).mean(),
}).reindex(BANDS))

In [ ]:
# 5c chart: survival curves by company volume (earlier listings only), with a Gehan–Wilcoxon test of 500+ vs. 1-4 listings
# Each curve is the Kaplan–Meier share of listings still open after each day; a median is where a curve crosses 50%.
# Days open is censored (still-open listings haven't finished), so the Wilcoxon test used here is the survival version:
# a log-rank test weighted toward early differences (Gehan–Breslow–Wilcoxon). Formal tests are in EDA_SDA_4; this is a preview.
# The test runs on up to 500,000 random listings per group; the curves use every listing.
from lifelines.statistics import logrank_test
TEST_N = 500_000
fig, ax = plt.subplots(figsize=(9, 4.2))
for band, color in zip(BANDS, [MUTED, AQUA, ORANGE, BLUE]):
    g = so.loc[asof_band == band, ["days_open", "live"]]
    if g.empty:
        continue
    curve = km_curve(g.days_open, ~g.live)
    ax.step(curve.index, 100 * curve.values, where="post", color=color, lw=1.8,
            label=f"{band} (median {km_median(g.days_open, ~g.live):.0f} days, {compact(len(g))} listings)")
ax.axhline(50, color=INK_2, ls="--", lw=0.8)
ax.text(1, 51.5, "median: half still open", color=INK_2, fontsize=8)
a, b = (so.loc[asof_band == band, ["days_open", "live"]] for band in ("500+", "1-4"))
a, b = (g.sample(min(len(g), TEST_N), random_state=0) for g in (a, b))
title = "Share still open by company volume (earlier listings in any field)"
if len(a) and len(b):
    test = logrank_test(a.days_open, b.days_open, event_observed_A=~a.live, event_observed_B=~b.live, weightings="wilcoxon")
    title += (f"\nGehan–Wilcoxon test, 500+ vs. 1-4 listings: statistic {test.test_statistic:,.0f}, "
              f"p {'< 0.001' if test.p_value < 0.001 else f'= {test.p_value:.3f}'}")
ax.set(title=title, xlabel="days since first seen", ylabel="% still open", xlim=(0, 120), ylim=(0, 100))
ax.legend(fontsize=8, frameon=False)
finish()

**What the data shows**
- **Company volume is the strongest timing signal:** counting only earlier listings in any field, scope listings at companies with 500+ listings close in a median **12 days, vs. 44 at companies with 1–4** (Gehan–Wilcoxon test, p < 0.001).
- **The gradient is steady across bands** (44, 34, 28 and 12 days) and holds for listings first seen from May (56 vs. 12 days).
- **High-volume employers dominate:** the top 10% of companies post 47–74% of each bucket's listings, and reposting rises with volume.

### 5d Source

Hiring platform, job board or company site: share, arrivals and window. Job boards are under-represented in the crawl.

In [ ]:
# 5d Sources: share of postings, arrival rate, window
show("5d. Source type", pd.DataFrame({"share": so.source_type.value_counts(normalize=True).round(3)}))
source = so.platform.astype(object).fillna(so.source_type.astype(object)).astype("category")  # platform, else source type
by_src = so[["days_open", "live", "full_week"]].groupby(source, observed=True).apply(lambda g: pd.Series({
    "postings": len(g),
    "share": len(g) / len(so),
    "new per week": g.full_week.sum() / FULL_WEEKS,
    "median days open (Kaplan–Meier)": km_median(g.days_open, ~g.live),
    "open after 7d": km_survival(g.days_open, ~g.live, [7])[0],
}))
show("5d. Platforms and other sources: volume, arrival rate, window", by_src.sort_values("postings", ascending=False).head(TOP))

In [ ]:
# 5d Source mix within each role bucket
mix = pd.crosstab(so.role_bucket, source, normalize="index").reindex(BUCKETS)
show_shaded("5d. Source mix within each role bucket (share of the bucket's listings)",
            mix[mix.sum().sort_values(ascending=False).index[:8]])

**What the data shows**
- **65% of scope listings are on company sites or other domains,** 28% on hiring platforms and 7.1% on job boards.
- **Windows differ widely by source:** LinkedIn, Oracle and Taleo listings close in a median 15 days, company sites 21, Workday 32, Greenhouse 49, Lever 66 and Ashby 71.
- **The mix is similar in every bucket** (company sites 56–70%), so source separates listings, not roles. Platform choice tracks employer size, so source is tested alongside company volume.

### 5e Competition: reposts

A repost is the same job listed again under a new id: how common, by bucket and source, and how soon after the original.

In [ ]:
# 5e Repost rates by role bucket and by source, and days from the original to each repost
def repost_rate(key):
    return final.groupby(key, observed=True).is_repost.agg(reposts="sum", repost_rate="mean")

show("5e. Repost rate by role bucket (every scope listing)", repost_rate("role_bucket").reindex(BUCKETS))
show("5e. Repost rate by source type", repost_rate("source_type"))
first = final[~final.is_repost & final.repost_fp.notna()].groupby("repost_fp").first_seen.min()
reposts = final.loc[final.is_repost, ["repost_fp", "first_seen"]].join(first.rename("original_seen"), on="repost_fp")
gap = (reposts.first_seen - reposts.original_seen).dt.total_seconds() / 86400
show("5e. Days from the original to each repost", gap.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).to_frame("days"))

**What the data shows**
- **6.5% of scope listings are reposts:** most in technical sales (9.4%) and business and data analysis (8.9%), fewest in quality assurance (4.6%) and IT support (4.9%).
- **Reposts come back fast:** a median 7.6 days after the original, and a quarter within a day, so many are re-listings or crawler refreshes rather than a failed hire.

### 5f Recruiters

Anonymous keys only; counts and shares.

In [ ]:
# 5f Recruiters (anonymous keys; aggregates only)
has_rec = so.recruiter_key.notna()
per_rec = so.recruiter_key.dropna().value_counts()
rec_band = pd.cut(cat_map(so.recruiter_key.astype("category"), per_rec) if has_rec.any() else pd.Series(np.nan, index=so.index),
                  [0, 1, 4, np.inf], labels=["1 listing", "2-4", "5+"])
show("5f. Recruiters in the scope", pd.DataFrame({"value": {
    "share of listings naming a recruiter": pct(has_rec.mean()),
    "recruiters (anonymous keys)": compact(per_rec.size),
    "recruiters with 5+ listings": compact((per_rec >= 5).sum()),
    "share of recruiter listings from recruiters with 5+": pct(per_rec[per_rec >= 5].sum() / max(per_rec.sum(), 1)),
    "share with a recruiter title": pct(so.recruiter_title.notna().mean()),
}}))
rows = {"named recruiter": has_rec, "no recruiter named": ~has_rec}
rows.update({f"recruiter with {b}": rec_band.eq(b) for b in ["1 listing", "2-4", "5+"]})
show("5f. Window by recruiter (Kaplan–Meier)", pd.DataFrame({name: {
    "listings": int(m.sum()),
    "median days open": km_median(so.days_open[m], ~so.live[m]) if m.any() else np.nan,
    "open after 7d": km_survival(so.days_open[m], ~so.live[m], [7])[0] if m.any() else np.nan,
} for name, m in rows.items()}).T)

**What the data shows**
- **Only 1.3% of scope listings name a recruiter:** 5,527 anonymous recruiters; the 437 with 5+ listings post 37% of recruiter listings.
- **Recruiter listings close much faster:** a median 11.3 days vs. 24.5 without one; for recruiters with 5+ listings, **6.5 days** (48% still open after a week vs. 72%).
- **Many recruiter listings may come from staffing agencies,** so EDA_SDA_2 tests recruiters against its staffing-agency flag before they're used.

### 5g Who is open to newcomers

The text rules behind the career-change flags, among listings with a description.

In [ ]:
# 5g Career-change accessibility: the text rules, by role bucket
def access(g):
    """Text measures for one group, among listings with a description."""
    d = g[g.has_desc]
    stated = d.years_required.dropna()
    return pd.Series({
        "states years required": d.years_required.notna().mean(),
        "median years required (where stated)": stated.median(),
        "asks 2 years or fewer (where stated)": (stated <= 2).mean(),
        "entry-level or training wording": d.entry_wording.mean(),
        "career-changer wording": d.changer_wording.mean(),
        "degree required": d.degree_required.mean(),
        "or equivalent experience": d.equivalent_experience.mean(),
        "median hourly pay (USD), open listings only": g.loc[g.career_open.fillna(False), "salary_hourly_usd"].median(),
    })

ACCESS_COLS = ["has_desc", "years_required", "entry_wording", "changer_wording", "degree_required",
               "equivalent_experience", "career_open", "salary_hourly_usd"]
acc = pd.DataFrame({b: access(so.loc[so.role_bucket == b, ACCESS_COLS]) for b in BUCKETS}).T
acc.loc["whole scope"] = access(so[ACCESS_COLS])
show_shaded("5g. Career-change text rules by role bucket (each column shaded on its own scale)", acc, highlight="whole scope")

**What the data shows**
- **IT support is the entry door:** 48% of its listings that state years ask for 2 or fewer, and 22% are entry level (5a). Then other computer roles (36% show any sign) and business and data analysis (33%). IT project management is the hardest (2.1% entry level, 9.1% open).
- **Open listings pay less, but still well:** a median $38.00 an hour, vs. $64.62 across the scope ($26.00 for open IT support listings). Degree requirements are rare (3.9%), but 61% of listings that state years ask for 5 or more.

### 5h Do the flags go with how long listings stay open?

Each row compares listings with and without a flag, among those where it's known. Keyword flags misfire, so EDA_SDA_3 checks them; urgency is left to its NLP.

In [ ]:
# 5h Do the flags relate to how long postings stay open?
flags = {
    "salary disclosed": so.salary_hourly_usd.notna(),
    "mentions remote": so.mentions_remote,
    "long description (top 25%)": so.desc_words > so.desc_words.quantile(0.75),
    "entry-level seniority": so.entry_seniority,
    "asks 2 years or fewer": (so.years_required <= 2).where(so.years_required.notna()).astype("boolean"),  # only listings that state years
    "asks 5 years or more": (so.years_required >= 5).where(so.years_required.notna()).astype("boolean"),
    "entry-level or training wording": so.entry_wording,
    "degree required": so.degree_required,
    "open to career changers (any sign)": so.career_open,
}
flag_rows = {}
for name, mask in flags.items():
    known = mask.notna().to_numpy()                  # compare with vs. known-without, never vs. unknown
    has = mask.fillna(False).astype(bool).to_numpy()
    with_f = km_survival(so.days_open[has], ~so.live[has], [7, 30])
    without = km_survival(so.days_open[known & ~has], ~so.live[known & ~has], [7, 30])
    flag_rows[name] = {"share of listings": has[known].mean(), "open 7d (with)": with_f[0], "open 7d (without)": without[0],
                       "open 30d (with)": with_f[1], "open 30d (without)": without[1]}
show("5h. Flags vs. application window (scope)", pd.DataFrame(flag_rows).T)

In [ ]:
# 5h chart: share still open after 7 days, with vs. without each flag
feat = pd.DataFrame(flag_rows).T.sort_values("open 7d (with)")
fig, ax = plt.subplots(figsize=(8.5, 4.4))
y = np.arange(len(feat))
ax.hlines(y, feat["open 7d (without)"], feat["open 7d (with)"], color=MUTED, lw=2)
ax.scatter(feat["open 7d (without)"], y, color=INK_2, s=60, zorder=3, label="without the feature")
ax.scatter(feat["open 7d (with)"], y, color=BLUE, s=60, zorder=3, label="with the feature")
ax.set_yticks(y, feat.index)
ax.set(title="Share of listings still open after 7 days", xlabel="share still open")
ax.grid(axis="y", visible=False)
ax.legend(fontsize=8.5, loc="lower right")
finish()

**What the data shows**
- **Disclosed pay and long descriptions go with listings staying open longer** (about 80% still open after a week, vs. about 65% without). **Remote wording shows no gap** (77% vs. 79%).
- **Listings open to career changers close a bit faster** (75% still open after a week vs. 79%).
- **One feature at a time:** these overlap with company volume and source, so EDA_SDA_4's Cox model separates them.

## 6. Q3: What time information can prioritization rely on?

Six months of data rules out seasons. Checks: history and weekly arrivals (full weeks only), whether `first_seen` follows the employer or the crawler, which listings count as still open, and whether the date picker is fair.

In [ ]:
# 6a How much history does the data hold, and how many listings arrive per full week?
weekly = so.first_seen.dt.tz_localize(None).dt.to_period("W").value_counts().sort_index()
full = weekly[(weekly.index > FIRST_WEEK) & (weekly.index < LAST_WEEK)]
show("6a. History and weekly arrivals (full weeks only)", pd.DataFrame({"value": {
    "first_seen from": so.first_seen.min().date(),
    "first_seen to": so.first_seen.max().date(),
    "distinct months": so.month.nunique(),
    "complete years of monthly data (seasonal decomposition needs 2+)": so.month.nunique() // 12,
    "full weeks": full.size,
    "median new listings per week": compact(full.median()),
    "lowest and highest week, vs. the median": f"{full.min() / full.median():.2f}× and {full.max() / full.median():.2f}×",
}}))

In [ ]:
# 6a chart: new listings per week (partial first and last weeks greyed, labelled)
fig, ax = plt.subplots(figsize=(9.5, 3.4))
ax.plot([p.start_time for p in full.index], full.values, color=BLUE, marker="o", ms=4, label="full week")
partial = weekly.drop(full.index)
ax.scatter([p.start_time for p in partial.index], partial.values, color=MUTED, s=40, zorder=3,
           label="partial week (left out of weekly rates)")
ax.set(title="New scope listings per week (first seen)", ylabel="listings")
ax.legend(fontsize=8.5)
finish()

In [ ]:
# 6b Employer or crawler? posted vs. first_seen
both = so.posted.notna()
lag = (so.first_seen - so.posted)[both].dt.total_seconds() / 86400
show(f"6b. Days from posted to first_seen (listings with posted: {pct(both.mean())})",
     lag.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(1).to_frame("days"))
days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday = pd.DataFrame({
    "posted (UTC)": so.posted[both].dt.day_name().value_counts(normalize=True),
    # posted timestamps sit at 22:00/23:00 UTC = midnight Central European time, so read them as CET dates
    "posted (CET date)": so.posted[both].dt.tz_convert("Europe/Berlin").dt.day_name().value_counts(normalize=True),
    "first_seen, same listings": so.first_seen[both].dt.day_name().value_counts(normalize=True),
    "first_seen, all listings": so.first_seen.dt.day_name().value_counts(normalize=True),
}).reindex(days)
show("6b. % of listings by weekday (if the columns disagree, first_seen reflects the crawler)",
     (100 * weekday).round(1))

In [ ]:
# 6c Which listings count as still open? Our rule vs. the hackathon app's (merge decision 2)
# Ours: last seen within 2 days of September 30. The app's signals: last seen within 7 days and not marked closed.
# The app's listing filter today: status not "closed" (status is only ever "closed" or empty).
days_since = (DATA_END - so.last_seen).dt.days
not_closed = so.status.ne("closed").fillna(True).astype(bool)
rules = {
    "ours: last seen within 2 days": so.live.astype(bool),
    "last seen within 7 days": days_since <= 7,
    "the app's signals: within 7 days and not closed": (days_since <= 7) & not_closed,
    "the app's listings today: status not closed": not_closed,
}
show("6c. Share of scope listings counted as still open, by rule", pd.DataFrame({
    "share counted open": {k: v.mean() for k, v in rules.items()},
    "share also open under ours (2 days)": {k: (v & rules["ours: last seen within 2 days"]).sum() / max(v.sum(), 1) for k, v in rules.items()},
    "share last seen 30+ days before Sep 30": {k: (v & (days_since > 30)).sum() / max(v.sum(), 1) for k, v in rules.items()},
}))

In [ ]:
# 6d Date picker check: scope listings live on the 1st of each month (shares and medians)
# Live on a date = first seen on or before it and last seen on or after it. The data starts on 2026-03-30, so listings
# that opened earlier are missing (left truncation): early dates should look thin and young.
rows = {}
for d in pd.date_range("2026-04-01", "2026-09-01", freq="MS", tz="UTC"):
    age = (d - so.first_seen[(so.first_seen <= d) & (so.last_seen >= d)]).dt.days
    rows[d.strftime("%Y-%m-%d")] = {"live listings": len(age), "median days since first seen": age.median(),
                                    "share first seen in the past 30 days": (age <= 30).mean()}
picker = pd.DataFrame(rows).T
picker.insert(1, "live, vs. September 1 (share)", picker["live listings"] / picker["live listings"].iloc[-1])
show("6d. Listings live on the 1st of each month", picker)
section_done("5–6. The scope explored")

**What the data shows**
- **Six months of steady history:** no seasonal advice, but 75% of scope listings closed by September 30 (25% were still open), so timing is well observed. A median **31K new scope listings arrive per full week**; the quietest week is 0.76× the median and the busiest 1.27×.
- **`first_seen` is a good freshness clock:** it lags the employer's posting date by a median 1.5 days (75% within 3.4). Its weekdays follow the crawler (Friday peak), while posting dates (41% of listings) peak Tuesday to Thursday.
- **Still-open rules (merge decision 2):** our rule (last seen within 2 days) counts 25% of scope listings as open, and the app's signal rule (7 days, not closed) 25%, mostly the same listings (94% overlap). **The app's current listing filter (status not closed) counts 39%, and 26% of those were last seen 30+ days before September 30.**
- **The date picker is fair from July:** live listings reach 80% of September's level, and the share first seen in the past 30 days settles near 50%.

## 7. The scope table (hand-off to EDA_SDA_2)

`data/clean/analysis_table.parquet`, with the full descriptions in `scope_descriptions.parquet` (joined by `id`); the dictionary lists every column.

In [ ]:
# 7a Save the scope table for EDA_SDA_2: the columns above, plus the text and list columns from the stream's rows (id,
# title, title_norm, job_title, desc_hash, contract_types with unified labels, tags, categories and the raw fields kept
# for EDA_SDA_2's feature screening), added part by part. EDA_SDA_2 reads only the columns it needs.
FINAL_PATH = CLEAN_DIR / "analysis_table.parquet"
start = time.time()
save_final(final, FINAL_PATH)
print(f"saved {FINAL_PATH.name} in {time.time() - start:.0f}s ({FINAL_PATH.stat().st_size / 1e9:.2f} GB, "
      f"{len(pq.read_schema(FINAL_PATH).names)} columns); descriptions in {DESC_PATH.name}")

In [ ]:
# 7b Feature dictionary: every column of the scope table, its role and what it means (src/feature_dictionary.py)
from src.feature_dictionary import feature_dictionary
dictionary, undocumented = feature_dictionary(final, FINAL_PATH)
show("Scope table: feature dictionary, by role (coverage = share of scope listings with a value)", dictionary)
if undocumented:
    print("columns without a description:", undocumented)
section_done("7. Scope table saved")

## 8. Findings

| | |
|:--|:--|
| **Data** | 13,415,388 listings in the file; 13,017,464 English (or unlabeled) and US (or no country); first seen March 30 to September 30, 2026 (the data's end date) |
| **After cleaning** | 13.0M listings, none removed; 1.4M reposts flagged (10%); 11.7M originals from 324K companies; 22.3% still open on September 30 |
| **Scope table** | 908K listings (849K originals) from 72K companies in 11 role buckets; 25% still open on September 30; 58% under a TripleTen program, 42% adjacent tech roles |

**Q1, how tech compares:** computer and mathematical is the best-paid (1st of 23) and most remote (1st) field, 4th in volume, but 22nd on entry-level roles and openness, and last on asking 2 years or fewer. The scope's design follows from it: hide what a newcomer can't get, rank the rest, and point to the open doors (IT support, business and data analysis, Data Analytics roles).

**Q2, which factors drive prioritization** (inside the scope)

- **Company volume is the strongest timing signal:** a median 12 days at companies with 500+ listings, vs. 44 at 1–4 (5c).
- **The window is short,** and it's what the model predicts: 29% of listings are gone within a week, median 24 days (5b).
- **Source and recruiters matter:** 15 days on LinkedIn or Taleo vs. 71 on Ashby; 11 days with a named recruiter, though only 1.3% of listings name one (5d, 5f).
- **Pay and openness vary most between roles:** $31–$74 an hour, and openness from 9% (IT project management) to 44% (IT support) (5a, 5g).
- **Weak, or flags only:** remote wording shows no gap, reposts are mostly re-listings, and crawl artifacts barely move timing (5b, 5e, 5h).

**Q3, time information:** posting age works for every listing (`first_seen` lags the posting date by a median 1.5 days); day of week only through `posted` (41%); no seasonality from six months. The still-open rule should be last-seen based: the app's status filter keeps 26% of listings last seen 30+ days before September 30 (6c).

### Limits
- **Six months of history,** and listings first seen before March 30 are missing, so early dates undercount what was open (the date picker starts July 1).
- **We see listings, not hires:** openness is what listings ask for.
- **The text flags are unvalidated keyword rules,** so they stay labels and opt-in filters.
- **Section 3 is descriptive;** EDA_SDA_4 tests the gaps and confounders (staffing agencies are flagged in EDA_SDA_2).

**Handoff:** the scope table, its descriptions, the occupation summary and company activity go to **EDA_SDA_2**.